# 04 決算データ収集 (Earnings / Financials)

| 項目 | 内容 |
|------|------|
| 目的 | yfinance を使って JP 株式の四半期決算データを収集し、Earnings Surprise 指標を算出 |
| 出力 | `data/earnings_financials.parquet`, `data/earnings_dates.parquet`, `data/ohlcv_with_earnings.parquet` |
| 注意 | J-Quants API は Phase 2 で導入予定。本ノートは yfinance で代替実装 |

### 収集データ一覧
| データ種別 | 取得元 | 内容 |
|----------|--------|------|
| 四半期損益計算書 | `quarterly_financials` | 売上高・営業利益・純利益・EPS |
| 四半期貸借対照表 | `quarterly_balance_sheet` | 総資産・純資産・有利子負債 |
| 四半期CF計算書 | `quarterly_cashflow` | 営業CF・設備投資・FCF |
| 決算発表日 | `calendar` / インデックス日付 | 決算発表日の代理変数 |

---
## 0. 環境セットアップ

In [ ]:
import warnings
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yfinance as yf
import yaml
from tqdm import tqdm

try:
    import japanize_matplotlib
except ImportError:
    print('japanize_matplotlib 未インストール。日本語フォントが文字化けする可能性あり')

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
%matplotlib inline

print(f'pandas  : {pd.__version__}')
print(f'yfinance: {yf.__version__}')
print('Setup OK')

---
## 1. 設定読み込み・シンボル定義・セクターマッピング

In [ ]:
# ---- 設定ファイル読み込み ----
CFG_PATH = Path('../../configs/stock_config.yaml')
with open(CFG_PATH, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

DATA_DIR      = Path(cfg['paths']['data'])
PROCESSED_DIR = Path(cfg['paths']['processed_data'])
FIGURES_DIR   = Path(cfg['paths']['figures'])
DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DATA_START     = cfg['equity']['data_start']
EQUITY_SYMBOLS = cfg['equity']['symbols']
RANDOM_SEED    = cfg.get('random_seed', 42)

print(f'Data start : {DATA_START}')
print(f'Equity     : {len(EQUITY_SYMBOLS)} 銘柄')
print(f'Data dir   : {DATA_DIR.resolve()}')

In [ ]:
# ---- 銘柄名・セクターマッピング (手動定義) ----
# config の equity.symbols に合わせて定義
SYMBOL_INFO = {
    '7203.T': {'name': 'トヨタ自動車',     'sector': '自動車',       'sector_en': 'Automobile'},
    '6758.T': {'name': 'ソニーグループ',   'sector': '電気機器',     'sector_en': 'Electronics'},
    '8306.T': {'name': '三菱UFJ FG',       'sector': '銀行',         'sector_en': 'Banking'},
    '9432.T': {'name': 'NTT',              'sector': '通信',         'sector_en': 'Telecom'},
    '9984.T': {'name': 'ソフトバンクG',    'sector': '通信',         'sector_en': 'Telecom'},
    '6861.T': {'name': 'キーエンス',       'sector': '電気機器',     'sector_en': 'Electronics'},
    '8316.T': {'name': '三井住友FG',       'sector': '銀行',         'sector_en': 'Banking'},
    '4063.T': {'name': '信越化学工業',     'sector': '化学',         'sector_en': 'Chemicals'},
    '6367.T': {'name': 'ダイキン工業',     'sector': '機械',         'sector_en': 'Machinery'},
    '7974.T': {'name': '任天堂',           'sector': 'その他製品',   'sector_en': 'Entertainment'},
    '8035.T': {'name': '東京エレクトロン', 'sector': '電気機器',     'sector_en': 'Electronics'},
    '6954.T': {'name': 'ファナック',       'sector': '電気機器',     'sector_en': 'Electronics'},
    '4519.T': {'name': '中外製薬',         'sector': '医薬品',       'sector_en': 'Pharma'},
    '9433.T': {'name': 'KDDI',             'sector': '通信',         'sector_en': 'Telecom'},
    '3382.T': {'name': 'セブン&アイ',      'sector': '小売業',       'sector_en': 'Retail'},
    '4502.T': {'name': '武田薬品工業',     'sector': '医薬品',       'sector_en': 'Pharma'},
    '7751.T': {'name': 'キヤノン',         'sector': '電気機器',     'sector_en': 'Electronics'},
    '6098.T': {'name': 'リクルートHD',     'sector': 'サービス業',   'sector_en': 'Services'},
    '8766.T': {'name': '東京海上HD',       'sector': '保険',         'sector_en': 'Insurance'},
    '6501.T': {'name': '日立製作所',       'sector': '電気機器',     'sector_en': 'Electronics'},
    '4568.T': {'name': '第一三共',         'sector': '医薬品',       'sector_en': 'Pharma'},
    '7267.T': {'name': '本田技研工業',     'sector': '自動車',       'sector_en': 'Automobile'},
    '6702.T': {'name': '富士通',           'sector': '電気機器',     'sector_en': 'Electronics'},
    '6503.T': {'name': '三菱電機',         'sector': '電気機器',     'sector_en': 'Electronics'},
    '5108.T': {'name': 'ブリヂストン',     'sector': 'ゴム製品',     'sector_en': 'Rubber'},
}

# config にない銘柄があれば Unknown で補完
for sym in EQUITY_SYMBOLS:
    if sym not in SYMBOL_INFO:
        SYMBOL_INFO[sym] = {'name': sym, 'sector': 'Unknown', 'sector_en': 'Unknown'}

df_meta = pd.DataFrame(SYMBOL_INFO).T
df_meta.index.name = 'symbol'
df_meta = df_meta.reset_index()
print(f'銘柄メタ: {len(df_meta)} 件')
print(df_meta['sector'].value_counts())
df_meta.head()

---
## 2. 四半期財務データ収集関数

In [ ]:
def safe_get_row(df: pd.DataFrame, keys: list):
    """DataFrame から複数のキー候補を順番に試して最初にヒットした行を返す。
    yfinance は API バージョンによってフィールド名が変わるため候補リストで対応。
    """
    if df is None or df.empty:
        return pd.Series(dtype=float)
    for k in keys:
        if k in df.index:
            return df.loc[k]
    return pd.Series(dtype=float)


def collect_quarterly_financials(symbol: str) -> dict:
    """1銘柄の四半期財務データを収集して dict of DataFrame を返す。

    Returns
    -------
    dict with keys:
        'income'   : 損益計算書系指標 DataFrame (index=date)
        'balance'  : 貸借対照表系指標 DataFrame (index=date)
        'cashflow' : CF計算書系指標 DataFrame (index=date)
        'error'    : None or str
    """
    result = {'income': None, 'balance': None, 'cashflow': None, 'error': None}
    try:
        tkr = yf.Ticker(symbol)

        # ---- 損益計算書 ----
        qfi = tkr.quarterly_financials  # columns=dates, index=items
        if qfi is not None and not qfi.empty:
            revenue = safe_get_row(qfi, [
                'Total Revenue', 'Revenue', 'Net Revenue',
                'TotalRevenue', 'Gross Profit',
            ])
            op_income = safe_get_row(qfi, [
                'Operating Income', 'EBIT', 'Operating Profit',
                'OperatingIncome',
            ])
            net_income = safe_get_row(qfi, [
                'Net Income', 'Net Income Common Stockholders',
                'NetIncome', 'Net Income From Continuing And Discontinued Operation',
            ])
            eps = safe_get_row(qfi, [
                'Basic EPS', 'Diluted EPS', 'EPS', 'Earnings Per Share',
            ])

            df_inc = pd.DataFrame({
                'revenue':    revenue,
                'op_income':  op_income,
                'net_income': net_income,
                'eps':        eps,
            })
            df_inc.index = pd.to_datetime(df_inc.index).tz_localize(None)
            df_inc = df_inc.sort_index()  # 昇順
            result['income'] = df_inc

        # ---- 貸借対照表 ----
        qbs = tkr.quarterly_balance_sheet
        if qbs is not None and not qbs.empty:
            total_assets = safe_get_row(qbs, [
                'Total Assets', 'TotalAssets',
            ])
            equity = safe_get_row(qbs, [
                'Stockholders Equity', 'Total Stockholder Equity',
                'Common Stock Equity', 'StockholdersEquity',
                'Total Equity Gross Minority Interest',
            ])
            total_debt = safe_get_row(qbs, [
                'Total Debt', 'Long Term Debt', 'TotalDebt',
                'Short Long Term Debt Total',
            ])

            df_bal = pd.DataFrame({
                'total_assets': total_assets,
                'equity':       equity,
                'total_debt':   total_debt,
            })
            df_bal.index = pd.to_datetime(df_bal.index).tz_localize(None)
            df_bal = df_bal.sort_index()
            result['balance'] = df_bal

        # ---- CF計算書 ----
        qcf = tkr.quarterly_cashflow
        if qcf is not None and not qcf.empty:
            op_cf = safe_get_row(qcf, [
                'Operating Cash Flow', 'Cash From Operations',
                'Cash Flow From Continuing Operating Activities',
                'OperatingCashFlow',
            ])
            capex = safe_get_row(qcf, [
                'Capital Expenditure', 'Capital Expenditures',
                'Purchase Of PPE', 'CapitalExpenditure',
            ])

            df_cf = pd.DataFrame({
                'op_cf': op_cf,
                'capex': capex,
            })
            df_cf.index = pd.to_datetime(df_cf.index).tz_localize(None)
            df_cf = df_cf.sort_index()
            # FCF = 営業CF + 設備投資 (capex は通常マイナス値)
            df_cf['fcf'] = df_cf['op_cf'].add(df_cf['capex'], fill_value=0)
            result['cashflow'] = df_cf

    except Exception as e:
        result['error'] = str(e)

    return result


print('collect_quarterly_financials 定義完了')

---
## 3. 全銘柄の四半期財務データ収集

In [ ]:
# ---- 全銘柄収集 (tqdm でプログレス表示) ----
financials_raw = {}  # symbol -> dict
errors = {}

for sym in tqdm(EQUITY_SYMBOLS, desc='財務データ収集'):
    res = collect_quarterly_financials(sym)
    financials_raw[sym] = res
    if res['error']:
        errors[sym] = res['error']

print(f'\n収集完了: {len(EQUITY_SYMBOLS)} 銘柄')
if errors:
    print(f'エラー発生: {len(errors)} 銘柄')
    for sym, err in errors.items():
        print(f'  {sym}: {err}')
else:
    print('エラーなし')

In [ ]:
# ---- YoY成長率・トレーリング4Q平均 を計算してフラットな DataFrame に結合 ----

records_inc = []
for sym, res in financials_raw.items():
    df_inc = res.get('income')
    df_bal = res.get('balance')
    df_cf  = res.get('cashflow')

    if df_inc is None or df_inc.empty:
        continue

    # 基本財務指標
    rec = df_inc.copy()
    rec['symbol'] = sym

    # YoY成長率 (4期前比) ── 4Q分のラグを取る
    for col in ['revenue', 'op_income', 'net_income']:
        if col in rec.columns:
            rec[f'{col}_yoy'] = rec[col].pct_change(4)

    # トレーリング4Q平均成長率 (サプライズ計算のベースライン)
    for col in ['revenue_yoy', 'op_income_yoy', 'net_income_yoy']:
        if col in rec.columns:
            rec[f'{col}_trail4q'] = rec[col].rolling(window=4, min_periods=2).mean()

    # 貸借対照表データを結合
    if df_bal is not None and not df_bal.empty:
        rec = rec.join(df_bal, how='left', rsuffix='_bal')

    # CF計算書データを結合
    if df_cf is not None and not df_cf.empty:
        rec = rec.join(df_cf, how='left', rsuffix='_cf')

    records_inc.append(rec)

if records_inc:
    df_financials = pd.concat(records_inc)
    df_financials.index.name = 'date'
    df_financials = df_financials.reset_index()
    df_financials['date'] = pd.to_datetime(df_financials['date']).dt.tz_localize(None)
    # メタ情報を結合
    df_financials = df_financials.merge(df_meta[['symbol', 'name', 'sector', 'sector_en']],
                                        on='symbol', how='left')
    df_financials = df_financials.sort_values(['symbol', 'date']).reset_index(drop=True)
    print(f'四半期財務 DataFrame: {df_financials.shape}')
    print(f'銘柄数: {df_financials["symbol"].nunique()}')
    print(f'期間  : {df_financials["date"].min().date()} ~ {df_financials["date"].max().date()}')
    df_financials.head()
else:
    print('警告: 財務データを取得できた銘柄がありません')
    df_financials = pd.DataFrame()

---
## 4. 決算発表日の抽出

In [ ]:
def get_earnings_dates(symbol: str) -> pd.DataFrame:
    """決算発表日を取得する。

    優先順位:
    1. yf.Ticker.get_earnings_dates() (過去分)
    2. quarterly_financials のインデックス日付 (代理変数)
    3. calendar の Earnings Date (次回予定)
    """
    rows = []
    try:
        tkr = yf.Ticker(symbol)

        # 方法1: get_earnings_dates (過去~将来の決算日一覧)
        try:
            ed = tkr.get_earnings_dates(limit=20)
            if ed is not None and not ed.empty:
                ed = ed.reset_index()
                date_col = ed.columns[0]  # 'Earnings Date' など
                for _, row in ed.iterrows():
                    dt = pd.to_datetime(row[date_col]).tz_localize(None) \
                         if hasattr(row[date_col], 'tzinfo') and row[date_col].tzinfo is None \
                         else pd.to_datetime(row[date_col]).tz_convert(None)
                    rows.append({'symbol': symbol, 'date': dt, 'source': 'earnings_dates'})
        except Exception:
            pass

        # 方法2: quarterly_financials のインデックス日付を代理変数として使用
        qfi = financials_raw.get(symbol, {}).get('income')
        if qfi is not None and not qfi.empty:
            for dt in qfi.index:
                # 実際の決算発表は期末から 1~3 ヶ月後が多い → 45 日後を近似
                announce_dt = dt + timedelta(days=45)
                rows.append({'symbol': symbol, 'date': announce_dt, 'source': 'qfi_proxy'})

        # 方法3: calendar の次回決算日
        try:
            cal = tkr.calendar
            if cal is not None:
                # dict 形式の場合
                if isinstance(cal, dict):
                    ed_val = cal.get('Earnings Date') or cal.get('earningsDate')
                    if ed_val is not None:
                        if hasattr(ed_val, '__iter__') and not isinstance(ed_val, str):
                            for v in ed_val:
                                rows.append({'symbol': symbol,
                                             'date': pd.to_datetime(v).tz_localize(None),
                                             'source': 'calendar'})
                        else:
                            rows.append({'symbol': symbol,
                                         'date': pd.to_datetime(ed_val).tz_localize(None),
                                         'source': 'calendar'})
                # DataFrame 形式の場合
                elif isinstance(cal, pd.DataFrame) and 'Earnings Date' in cal.index:
                    for v in cal.loc['Earnings Date']:
                        rows.append({'symbol': symbol,
                                     'date': pd.to_datetime(v).tz_localize(None),
                                     'source': 'calendar'})
        except Exception:
            pass

    except Exception as e:
        pass

    if not rows:
        return pd.DataFrame(columns=['symbol', 'date', 'source'])

    df = pd.DataFrame(rows)
    df['date'] = pd.to_datetime(df['date']).dt.tz_localize(None)
    df = df.dropna(subset=['date'])
    df = df.drop_duplicates(subset=['symbol', 'date'])
    # 四半期ラベル付与
    df['year']  = df['date'].dt.year
    df['qtr']   = df['date'].dt.quarter
    df['qtr_label'] = df['year'].astype(str) + 'Q' + df['qtr'].astype(str)
    return df.sort_values('date').reset_index(drop=True)


print('get_earnings_dates 定義完了')

In [ ]:
# ---- 全銘柄の決算日収集 ----
date_records = []
for sym in tqdm(EQUITY_SYMBOLS, desc='決算日収集'):
    df_dates = get_earnings_dates(sym)
    if not df_dates.empty:
        date_records.append(df_dates)

if date_records:
    df_earnings_dates = pd.concat(date_records, ignore_index=True)
    df_earnings_dates = df_earnings_dates.sort_values(['symbol', 'date']).reset_index(drop=True)
    print(f'\n決算日 DataFrame: {df_earnings_dates.shape}')
    print(f'銘柄数: {df_earnings_dates["symbol"].nunique()}')
    print(f'ソース別件数:')
    print(df_earnings_dates['source'].value_counts())
    df_earnings_dates.head(10)
else:
    print('警告: 決算日データを取得できませんでした')
    df_earnings_dates = pd.DataFrame(columns=['symbol', 'date', 'source', 'year', 'qtr', 'qtr_label'])

---
## 5. Earnings Surprise 指標の算出

In [ ]:
def compute_surprise_metrics(df: pd.DataFrame) -> pd.DataFrame:
    """クロスセクショナルに Surprise 指標を算出する。

    Surprise = 当期 YoY 成長率 - トレーリング4Q平均成長率
    → 各四半期末時点でクロスセクションに Z スコア正規化
    """
    df = df.copy()

    # Revenue Surprise
    if 'revenue_yoy' in df.columns and 'revenue_yoy_trail4q' in df.columns:
        df['revenue_surprise_raw'] = df['revenue_yoy'] - df['revenue_yoy_trail4q']

    # Net Income Surprise
    if 'net_income_yoy' in df.columns and 'net_income_yoy_trail4q' in df.columns:
        df['ni_surprise_raw'] = df['net_income_yoy'] - df['net_income_yoy_trail4q']

    # クロスセクション Z スコア正規化 (date ごとに正規化)
    for col_raw, col_z in [
        ('revenue_surprise_raw', 'revenue_surprise_z'),
        ('ni_surprise_raw',      'ni_surprise_z'),
    ]:
        if col_raw not in df.columns:
            continue
        df[col_z] = df.groupby('date')[col_raw].transform(
            lambda x: (x - x.mean()) / (x.std() + 1e-8)
        )

    return df


if not df_financials.empty:
    df_financials = compute_surprise_metrics(df_financials)

    surprise_cols = [c for c in df_financials.columns if 'surprise' in c]
    print('Surprise 指標列:')
    for c in surprise_cols:
        n_valid = df_financials[c].notna().sum()
        print(f'  {c}: {n_valid} 件')

    # Earnings Dates DataFrame に Surprise 指標を結合
    if not df_earnings_dates.empty:
        # quarterly_financials の date と earnings_dates を近似マージ
        # proxy の場合は qfi の date + 45 日 → qfi の date で逆算してマージ
        df_fin_slim = df_financials[[
            'symbol', 'date', 'revenue_yoy', 'net_income_yoy',
            'revenue_surprise_raw', 'revenue_surprise_z',
            'ni_surprise_raw', 'ni_surprise_z',
        ]].copy()
        # qfi_proxy は date = 期末 + 45 日。期末 ≈ earnings_dates.date - 45 日
        df_earnings_dates['fin_date_approx'] = df_earnings_dates['date'] - timedelta(days=45)

        # merge_asof で最近傍マージ (tolerance=60日)
        df_fin_slim_sorted = df_fin_slim.sort_values('date')
        ed_sorted = df_earnings_dates.sort_values('date')

        merged_parts = []
        for sym in EQUITY_SYMBOLS:
            ed_sym  = ed_sorted[ed_sorted['symbol'] == sym].copy()
            fin_sym = df_fin_slim_sorted[df_fin_slim_sorted['symbol'] == sym].copy()
            if ed_sym.empty or fin_sym.empty:
                merged_parts.append(ed_sym)
                continue
            merged = pd.merge_asof(
                ed_sym.sort_values('fin_date_approx'),
                fin_sym.sort_values('date').rename(columns={'date': 'fin_date'}),
                left_on='fin_date_approx',
                right_on='fin_date',
                by='symbol',
                tolerance=pd.Timedelta('60D'),
                direction='nearest',
            )
            merged_parts.append(merged)

        if merged_parts:
            df_earnings_dates = pd.concat(merged_parts, ignore_index=True)
            df_earnings_dates = df_earnings_dates.sort_values(['symbol', 'date']).reset_index(drop=True)

    print('\nSurprise 指標サンプル:')
    display_cols = ['symbol', 'date', 'revenue_surprise_raw', 'revenue_surprise_z',
                    'ni_surprise_raw', 'ni_surprise_z']
    available = [c for c in display_cols if c in df_financials.columns]
    print(df_financials[available].dropna().head(10))
else:
    print('警告: df_financials が空のため Surprise 計算をスキップ')

---
## 6. Earnings EDA (探索的データ分析)

In [ ]:
# ---- 6-1. 売上高 YoY 成長率のヒストグラム ----
if not df_financials.empty and 'revenue_yoy' in df_financials.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    data_yoy = df_financials['revenue_yoy'].dropna()
    # 外れ値を除外して表示 (±200%)
    data_yoy_clipped = data_yoy.clip(-2.0, 2.0)
    ax.hist(data_yoy_clipped, bins=40, color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(0, color='red', linestyle='--', linewidth=1.5, label='YoY=0%')
    ax.axvline(data_yoy.median(), color='orange', linestyle='--', linewidth=1.5,
               label=f'中央値: {data_yoy.median():.1%}')
    ax.set_xlabel('売上高 YoY 成長率')
    ax.set_ylabel('頻度')
    ax.set_title('売上高 YoY 成長率の分布 (全銘柄・全期間、±200%クリップ)')
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    save_path = FIGURES_DIR / '04_revenue_yoy_hist.png'
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'保存: {save_path}')
else:
    print('revenue_yoy データなし: スキップ')

In [ ]:
# ---- 6-2. セクター別 売上高 YoY 成長率 箱ひげ図 ----
if not df_financials.empty and 'revenue_yoy' in df_financials.columns and 'sector' in df_financials.columns:
    sector_data = {}
    for sec, grp in df_financials.groupby('sector'):
        vals = grp['revenue_yoy'].dropna().clip(-1.5, 1.5).values
        if len(vals) >= 3:
            sector_data[sec] = vals

    if sector_data:
        sectors_sorted = sorted(sector_data.keys(),
                                key=lambda s: np.median(sector_data[s]),
                                reverse=True)
        fig, ax = plt.subplots(figsize=(12, 6))
        bp = ax.boxplot(
            [sector_data[s] for s in sectors_sorted],
            labels=sectors_sorted,
            patch_artist=True,
            medianprops=dict(color='red', linewidth=2),
        )
        colors = plt.cm.Set3(np.linspace(0, 1, len(sectors_sorted)))
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
        ax.axhline(0, color='gray', linestyle='--', linewidth=1)
        ax.set_xticklabels(sectors_sorted, rotation=30, ha='right')
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
        ax.set_ylabel('売上高 YoY 成長率')
        ax.set_title('セクター別 売上高 YoY 成長率')
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        save_path = FIGURES_DIR / '04_sector_revenue_yoy_box.png'
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        plt.show()
        print(f'保存: {save_path}')
    else:
        print('セクター別データ不足: スキップ')
else:
    print('データなし: スキップ')

In [ ]:
# ---- 6-3. 上位5銘柄の売上高トレンド (折れ線グラフ) ----
if not df_financials.empty and 'revenue' in df_financials.columns:
    # 最新期末の売上高が大きい銘柄 Top 5 を選択
    latest_rev = (
        df_financials
        .dropna(subset=['revenue'])
        .sort_values('date')
        .groupby('symbol')
        .last()
        ['revenue']
        .nlargest(5)
    )
    top5 = latest_rev.index.tolist()

    fig, ax = plt.subplots(figsize=(14, 6))
    colors = plt.cm.tab10(np.linspace(0, 0.5, len(top5)))
    for sym, color in zip(top5, colors):
        df_s = df_financials[df_financials['symbol'] == sym].dropna(subset=['revenue'])
        name = SYMBOL_INFO.get(sym, {}).get('name', sym)
        ax.plot(df_s['date'], df_s['revenue'] / 1e9,
                marker='o', markersize=4, linewidth=1.5,
                label=f'{sym} ({name})', color=color)

    ax.set_xlabel('四半期末')
    ax.set_ylabel('売上高 (十億円/USD)')
    ax.set_title('売上高トレンド (Top 5 銘柄, 四半期)')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.xticks(rotation=30)
    plt.tight_layout()
    save_path = FIGURES_DIR / '04_top5_revenue_trend.png'
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Top5: {top5}')
    print(f'保存: {save_path}')
else:
    print('revenue データなし: スキップ')

---
## 7. OHLCV データとのマージ

In [ ]:
# ---- 7-1. OHLCV 読み込み ----
ohlcv_path = DATA_DIR / 'equity_jp_ohlcv.parquet'

if ohlcv_path.exists():
    df_ohlcv = pd.read_parquet(ohlcv_path)
    df_ohlcv['date'] = pd.to_datetime(df_ohlcv['date']).dt.tz_localize(None)
    print(f'OHLCV 読み込み: {df_ohlcv.shape}')
    print(f'期間: {df_ohlcv["date"].min().date()} ~ {df_ohlcv["date"].max().date()}')
    df_ohlcv.head()
else:
    print(f'警告: {ohlcv_path} が見つかりません。先に 03_equity_data_collection.ipynb を実行してください。')
    df_ohlcv = pd.DataFrame()

In [ ]:
# ---- 7-2. earnings_flag・決算前リターンの付与 ----

def merge_earnings_to_ohlcv(df_ohlcv: pd.DataFrame,
                             df_ed: pd.DataFrame,
                             df_fin: pd.DataFrame) -> pd.DataFrame:
    """OHLCV に決算フラグと決算前リターン (センチメント) を付与する。

    - earnings_flag: 決算発表日の最近傍取引日に 1 を立てる
    - pre5d_ret: 決算発表日の 5 営業日前リターン
    - pre20d_ret: 決算発表日の 20 営業日前リターン
    - surprise 指標: earnings_dates から引き継ぐ
    """
    if df_ohlcv.empty:
        return df_ohlcv

    df = df_ohlcv.copy().sort_values(['symbol', 'date'])

    # 決算フラグ列を初期化
    df['earnings_flag']      = 0
    df['pre5d_ret']          = np.nan
    df['pre20d_ret']         = np.nan
    df['revenue_surprise_z'] = np.nan
    df['ni_surprise_z']      = np.nan

    for sym in tqdm(EQUITY_SYMBOLS, desc='フラグ付与'):
        mask_ohlcv = df['symbol'] == sym
        df_sym = df[mask_ohlcv].copy()
        if df_sym.empty:
            continue

        # 該当銘柄の決算日リスト
        ed_sym = df_ed[df_ed['symbol'] == sym].copy() if not df_ed.empty else pd.DataFrame()

        if ed_sym.empty:
            continue

        trading_dates = df_sym['date'].values  # numpy datetime64
        close_series  = df_sym.set_index('date')['close']

        for _, ed_row in ed_sym.iterrows():
            earn_dt = ed_row['date']

            # 最近傍取引日を検索 (前後 5 営業日以内)
            diff = np.abs(
                pd.to_datetime(trading_dates) - earn_dt
            ).dt.days.values if hasattr(
                pd.to_datetime(trading_dates) - earn_dt, 'dt'
            ) else np.abs([(pd.Timestamp(d) - earn_dt).days for d in trading_dates])

            if diff.min() > 5:
                continue
            nearest_idx = df_sym.index[diff.argmin()]
            df.loc[nearest_idx, 'earnings_flag'] = 1

            # 決算発表直前の株価リターン (センチメント代理)
            nearest_date = df_sym.loc[nearest_idx, 'date']
            try:
                # 5 営業日前リターン
                before_5d = nearest_date - timedelta(days=7)  # カレンダー7日 ≈ 5営業日
                s5 = close_series.loc[before_5d:nearest_date]
                if len(s5) >= 2:
                    df.loc[nearest_idx, 'pre5d_ret'] = s5.iloc[-1] / s5.iloc[0] - 1

                # 20 営業日前リターン
                before_20d = nearest_date - timedelta(days=30)  # ≈ 20営業日
                s20 = close_series.loc[before_20d:nearest_date]
                if len(s20) >= 2:
                    df.loc[nearest_idx, 'pre20d_ret'] = s20.iloc[-1] / s20.iloc[0] - 1
            except Exception:
                pass

            # Surprise 指標の引き継ぎ
            for col in ['revenue_surprise_z', 'ni_surprise_z']:
                if col in ed_row.index and pd.notna(ed_row.get(col)):
                    df.loc[nearest_idx, col] = ed_row[col]

    return df.sort_values(['symbol', 'date']).reset_index(drop=True)


if not df_ohlcv.empty:
    df_ohlcv_merged = merge_earnings_to_ohlcv(df_ohlcv, df_earnings_dates, df_financials)
    n_flags = df_ohlcv_merged['earnings_flag'].sum()
    print(f'\nearnings_flag=1 の行数: {n_flags:,}')
    print(f'出力 DataFrame: {df_ohlcv_merged.shape}')
    df_ohlcv_merged[df_ohlcv_merged['earnings_flag'] == 1][
        ['symbol', 'date', 'close', 'earnings_flag', 'pre5d_ret', 'pre20d_ret']
    ].head(10)
else:
    print('OHLCV データがないためマージをスキップ')
    df_ohlcv_merged = pd.DataFrame()

---
## 8. データ保存

In [ ]:
# ---- 8-1. earnings_financials.parquet ----
if not df_financials.empty:
    out_path = DATA_DIR / 'earnings_financials.parquet'
    df_financials.to_parquet(out_path, index=False)
    print(f'保存: {out_path}  ({len(df_financials):,} 行)')
else:
    print('警告: df_financials が空のため保存をスキップ')

# ---- 8-2. earnings_dates.parquet ----
if not df_earnings_dates.empty:
    out_path = DATA_DIR / 'earnings_dates.parquet'
    df_earnings_dates.to_parquet(out_path, index=False)
    print(f'保存: {out_path}  ({len(df_earnings_dates):,} 行)')
else:
    print('警告: df_earnings_dates が空のため保存をスキップ')

# ---- 8-3. ohlcv_with_earnings.parquet ----
if not df_ohlcv_merged.empty:
    out_path = DATA_DIR / 'ohlcv_with_earnings.parquet'
    df_ohlcv_merged.to_parquet(out_path, index=False)
    print(f'保存: {out_path}  ({len(df_ohlcv_merged):,} 行)')
else:
    print('警告: ohlcv_with_earnings が空のため保存をスキップ')

print('\n全保存完了')

---
## 9. データ品質サマリー

In [ ]:
# ---- 9-1. 銘柄別カバレッジ (四半期財務) ----
print('=' * 60)
print('銘柄別 財務データ カバレッジ')
print('=' * 60)

coverage_rows = []
for sym in EQUITY_SYMBOLS:
    res = financials_raw.get(sym, {})
    inc = res.get('income')
    bal = res.get('balance')
    cf  = res.get('cashflow')
    err = res.get('error')
    name = SYMBOL_INFO.get(sym, {}).get('name', sym)

    row = {
        'symbol':     sym,
        'name':       name,
        'inc_qtrs':   len(inc) if inc is not None and not inc.empty else 0,
        'bal_qtrs':   len(bal) if bal is not None and not bal.empty else 0,
        'cf_qtrs':    len(cf)  if cf  is not None and not cf.empty  else 0,
        'error':      '✗' if err else '✓',
    }
    coverage_rows.append(row)

df_coverage = pd.DataFrame(coverage_rows)
print(df_coverage.to_string(index=False))

In [ ]:
# ---- 9-2. 欠損率レポート (四半期財務 DataFrame) ----
print('=' * 60)
print('四半期財務 DataFrame 欠損率')
print('=' * 60)

if not df_financials.empty:
    numeric_cols = df_financials.select_dtypes(include='number').columns.tolist()
    missing_rate = (df_financials[numeric_cols].isnull().mean() * 100).sort_values(ascending=False)
    print(missing_rate.map(lambda x: f'{x:.1f}%').to_string())
else:
    print('データなし')

print()
print('=' * 60)
print('統計サマリー')
print('=' * 60)
print(f'収集成功銘柄数  : {df_coverage[df_coverage["inc_qtrs"] > 0].shape[0]} / {len(EQUITY_SYMBOLS)}')
if not df_financials.empty:
    print(f'総四半期レコード: {len(df_financials):,}')
    print(f'平均四半期数/銘柄: {df_financials.groupby("symbol").size().mean():.1f}')
    print(f'期間: {df_financials["date"].min().date()} ~ {df_financials["date"].max().date()}')
if not df_earnings_dates.empty:
    print(f'決算日レコード  : {len(df_earnings_dates):,}')
if not df_ohlcv_merged.empty:
    earnings_days = df_ohlcv_merged['earnings_flag'].sum()
    print(f'earnings_flag=1 : {int(earnings_days):,} 日')

In [ ]:
# ---- 9-3. カバレッジ可視化 ----
fig, ax = plt.subplots(figsize=(12, 7))

df_cov_sorted = df_coverage.sort_values('inc_qtrs', ascending=True)
y_pos = range(len(df_cov_sorted))
ax.barh(y_pos, df_cov_sorted['inc_qtrs'], color='steelblue',  alpha=0.8, label='損益計算書')
ax.barh(y_pos, df_cov_sorted['bal_qtrs'], color='darkorange', alpha=0.6, label='貸借対照表')
ax.barh(y_pos, df_cov_sorted['cf_qtrs'],  color='green',      alpha=0.4, label='CF計算書')
ax.set_yticks(list(y_pos))
ax.set_yticklabels(
    [f"{r['symbol']} ({r['name']})" for _, r in df_cov_sorted.iterrows()],
    fontsize=9
)
ax.set_xlabel('四半期数')
ax.set_title('銘柄別 財務データ カバレッジ (四半期数)')
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
save_path = FIGURES_DIR / '04_coverage_summary.png'
plt.savefig(save_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'保存: {save_path}')
print('\n次のステップ: 42_equity_lgbm_walkforward.ipynb で決算特徴量を組み込む')